# 13 · Nested spatial cross-validation (5-class)

| | |
|---|---|
| Models | RF, XGBoost, MLP, GWRF, GWXGB, plus the depth-slice class shares as a non-competing reference (`DEPTH_PRIOR`) |
| Inputs | exactly `UTM_Easting_m`, `UTM_Northing_m`, `Top_Elevation_m` |
| Outer folds | the R blockCV folds with the block size from the autocorrelation range (notebook 12) |
| Training | Downtown development boreholes outside the test fold (scenario A) |
| Inner folds | the four remaining outer folds (R fallback if a class is missing from one) |
| Tuning | Optuna TPE with a fixed seed, maximising pooled inner out-of-fold **Macro-F1** over the 5 classes; the prediction is the most probable class |
| Preprocessing | MLP scaling and balanced class weights are computed inside `fit` (training partition only) |
| Checkpoints | one file per outer fold in `outputs_5class/03_nested_cv/checkpoints/`; re-running skips finished folds |

**GW models for 5 classes.** As in the binary project, every training borehole is an anchor with a local model, trained on its *k* nearest boreholes and blended with a global model (*p* = α *p*<sub>local</sub> + (1 − α) *p*<sub>global</sub>). A neighbourhood needs at least 5 boreholes and two classes with at least 10 samples. A class missing from a neighbourhood gets probability 0 from the local model and is supplied by the global model through the blend.

In [ ]:
import os, sys, json
from pathlib import Path

# 5-class soil model: this line selects config/project_5class.toml (keep it).
os.environ["SIEMREAP_CONFIG"] = "project_5class.toml"
# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner(), "|", cfg["_config_name"])

from siemreap import nested_cv as NCV, folds as F
from siemreap import data as D, plots as PL
CLS = D.class_info(cfg)["classes"]
NAMES = [c["name"] for c in CLS]
COLOURS = PL.class_colours(CLS)
K = len(NAMES)
OUT = config.stage_dir(cfg, "03_nested_cv", "")
AGG = config.stage_dir(cfg, "03_nested_cv", "aggregate")
RES = config.stage_dir(cfg, "03_nested_cv", "restricted")
mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")
print("classes:", NAMES)
print("trials per outer fold:", cfg["smoke"]["n_trials"] if MODE == "smoke" else cfg["experiment"]["n_trials"])

## Run specification

In [ ]:
specs = NCV.specs_from_config(cfg, MODE)
outer_folds = cfg["smoke"]["outer_folds"] if MODE == "smoke" else list(range(1, cfg["blockcv"]["k"] + 1))
RUN_MODELS = None      # e.g. ["RF", "XGB"] to run part of the experiment first (a list, in brackets)
specs = [s for s in specs if RUN_MODELS is None or s.model in RUN_MODELS + ["DEPTH_PRIOR"]]
print(len(specs), "runs x", len(outer_folds), "outer folds")
display(pd.DataFrame([s.__dict__ for s in specs]))

## Run (resumable)

In [ ]:
import time
primary = F.primary_block(cfg) if MODE == "full" else specs[0].block_m
all_preds, all_info = [], []
t0 = time.time()
for spec in specs:
    keep = spec.block_m == primary and spec.purge_m == 0
    p, info = NCV.run(mt, cfg, spec, outer_folds, MODE, OUT, save_models=keep)
    all_preds.append(p); all_info += info
print(f"done in {(time.time() - t0) / 60:.1f} min")

## Save predictions and run records

In [ ]:
preds = pd.concat(all_preds, ignore_index=True)
info = pd.DataFrame(all_info)
preds.to_csv(RES / "outer_predictions.csv", index=False)            # borehole codes and coordinates: restricted
info_public = info.drop(columns=["inner_adequacy"])
info_public.to_csv(AGG / "run_info.csv", index=False)
(AGG / "environment.json").write_text(json.dumps(config.environment_record(), indent=1))
display(info_public.groupby(["block_m", "model"])[["inner_macro_f1", "n_trials", "seconds"]].mean().round(3))
print("inner-fold source:", info.inner_source.value_counts().to_dict())

## Determinism check (one outer fold of RF and XGBoost re-fitted from scratch)

In [ ]:
pcols = [f"p_{c}" for c in range(K)]
for s in [s for s in specs if s.model in ("RF", "XGB")][:2]:
    k = outer_folds[0]
    again = NCV.run_fold(mt, F.load_outer(cfg, s.block_m), s, k, cfg, MODE)["predictions"]
    stored = preds[(preds.model == s.model) & (preds.block_m == s.block_m) & (preds.outer_fold == k)]
    same = np.array_equal(again[pcols].values, stored[pcols].values)
    print(s.tag, "fold", k, "identical predictions:", same)
    assert same